# Introduction: Accessing Bloomberg Data with Python

In this notebook, we'll walk through the **step-by-step process** of accessing financial data from the **Bloomberg Terminal using Python**.


We'll cover
- What libraries you need to get started
- How to open a session with Bloomberg
- How to find tickers and fields
- How to request and store data in a DataFrame
- Common issues and download limits to be aware of

This is a simplified demo to help you get started. Once you’re comfortable with the workflow, you can extend it to whatever data you need—equities, currencies, bonds, commodities, or macro indicators.

## Step 0: Logging in to Bloomberg

To use the Bloomberg API in Python, you'll need to be **logged into a Bloomberg Terminal** on one of the university’s designated Bloomberg computers.

> **The Python connection only works if Bloomberg is actively running on the machine you're using.**

### Where to Access Bloomberg Terminals
- There are **multiple Bloomberg machines available in the Business School**.
- You can also **book a Bloomberg computer** in advance using the library's room booking system:

🔗 [Book a Bloomberg computer](https://www.imperial.ac.uk/admin-services/library/use-the-library/room-bookings/)

If you're new to Bloomberg, this guide will help you get started

🔗 [Getting started with Bloomberg at Imperial](https://www.imperial.ac.uk/admin-services/library/subject-support/business/bloomberg/)

### Explore Bloomberg Using the Terminal

Once you're logged in to a Bloomberg Terminal, we recommend spending a few minutes exploring the interface and getting comfortable with the platform.

The **“Getting Started with Bloomberg at Imperial”** guide also includes

- Common Bloomberg Terminal functions  
- How to search for tickers and securities  
- Pages like `CTM <GO>`, `FLDS <GO>`, and `DES <GO>`  
- Basic navigation tips  

🔗 [Explore the guide](https://www.imperial.ac.uk/admin-services/library/subject-support/business/bloomberg/)

This will help you find tickers and data types that you'll later use in Python. You don’t need to memorise codes — Bloomberg makes it easy to search and browse.

## Step 1: Install the Bloomberg API for Python

To extract data from Bloomberg using Python, we’ll be using Bloomberg’s **official API library** called `blpapi`.

> **Important:** You must be logged in and have the Bloomberg Terminal running on the same machine when requesting data via Python.

You can find the Bloomberg API library and documentation here   
🔗 [Bloomberg API Library](https://www.bloomberg.com/professional/support/api-library/)

### Install the `blpapi` package

On the Bloomberg Terminal machine, open a command prompt or use a Python notebook and run
```python
pip install --index-url=https://blpapi.bloomberg.com/repository/releases/python/simple/ blpapi
```

In [ ]:
pip install --index-url=https://blpapi.bloomberg.com/repository/releases/python/simple/ blpapi

# uv add blpapi --index "https://blpapi.bloomberg.com/repository/releases/python/simple/"

Note: you may need to restart the kernel to use updated packages.


c:\Users\jj424\projects\aqms\.venv\Scripts\python.exe: No module named pip


## Step 2: Import Libraries and Start a Bloomberg Session

Now that the Bloomberg API is installed, we can connect to the Terminal using Python.

This block imports the necessary libraries and sets up a session to communicate with Bloomberg. Make sure the Terminal is running before executing this step.

In [3]:
import blpapi
from blpapi import SessionOptions, Session, Request
import pandas as pd
import datetime as dt

# Bloomberg session setup
session_options = SessionOptions()
session_options.setServerHost('localhost')
session_options.setServerPort(8194)
session = Session(session_options)

if not session.start():
    print("Failed to start session.")
    exit()

if not session.openService("//blp/refdata"):
    print("Failed to open Bloomberg refdata service.")
    exit()

c:\Users\jj424\projects\aqms\.venv\Lib\site-packages\blpapi\resolutionlist.py:86: SyntaxWarning: invalid escape sequence '\s'
  InvalidArgumentException: If adding :class:`Message`\s received
c:\Users\jj424\projects\aqms\.venv\Lib\site-packages\blpapi\topiclist.py:93: SyntaxWarning: invalid escape sequence '\s'
  InvalidArgumentException: If adding :class:`Message`\s received


### What's Happening Here?

Let’s break down what this code is doing

- `import blpapi` – Loads the Bloomberg API library.
- `SessionOptions()` – Creates a configuration object for the connection.
- `session_options.setServerHost('localhost')` – Tells the API to connect to the Bloomberg Terminal running on the **local machine**.
- `session_options.setServerPort(8194)` – Uses Bloomberg's default API port (**8194**) for communication.
- `Session(session_options)` – Creates a session using the settings you just defined.
- `session.start()` – Starts the session. If Bloomberg isn’t open, this will fail.
- `session.openService("//blp/refdata")` – Opens the **Reference Data Service**, which is used to request historical prices, volumes, open interest, etc.

> If either the session or the service can’t be opened, we print an error and stop the program using `exit()`.

This sets up the connection — next, we’ll use this session to request specific data from Bloomberg.

## Step 3: Finding Tickers and Data Fields

To request data from Bloomberg, you need to know two things
1. The **ticker symbol** (e.g. `AAPL US Equity`, `TYM24 Comdty`)
2. The **field name** (e.g. `PX_LAST` for price, `VOLUME`, `OPEN_INT`)

---

### Where to Find Tickers and Field Names

#### Option 1: Use the Bloomberg Terminal
While you're logged in to a Bloomberg machine, you can use these commands:

| Command      | What it Does                             |
|--------------|------------------------------------------|
| `CTM <GO>`   | Browse contracts (futures, FX, etc.)     |
| `FLDS <GO>`  | Search for available field names         |
| `DES <GO>`   | View full details of a security          |
| `WEI <GO>`   | See intraday and historical data options |

> Example: Type `TYM24 <F8>` and press `GO` to view the US 10-Year Treasury June 2024 futures contract.

#### Option 2: Use Bloomberg’s Website  
If you’re not at a terminal, you can still explore major securities via Bloomberg’s online market listings:

🔗 [Browse tickers on Bloomberg Markets](https://www.bloomberg.com/markets/stocks)

> This page is especially useful for **stock indices**, **major futures contracts**, and **company tickers**.

---

Once you have a few tickers and field names in mind, you're ready to build your first request!

## Step 4: Request G10 Currency Futures Data

Now that we’re connected to Bloomberg and know how to find tickers and fields, we can build a full request.

In this example, we’ll download **daily data** for **G10 currency futures** from the year 2000 to today.

We’ll request **Last price** (`PX_LAST`), **Trading volume** (`VOLUME`), **Open interest** (`OPEN_INT`)

> All contracts are quoted vs USD (e.g., EUR/USD, GBP/USD, JPY/USD). Since USD is the base, we only pull the other 9 G10 currencies.

In [5]:
# Define G10 futures codes (excluding USD)
currencies = {
    'EUR': 'EC',
    'JPY': 'JY',
    'GBP': 'BP',
    'AUD': 'AD',
    'NZD': 'NV',
    'CAD': 'CD',
    'CHF': 'SF',
    'NOK': 'NO',
    'SEK': 'SE'
}

# Define standard futures expiry month codes
month_codes = ['H', 'M', 'U', 'Z']  # March, June, September, December

# Create a list of tickers (e.g., "ECH10 Curncy" for March 2010 EUR/USD)
years = range(2000, 2026)
tickers = [f"{code}{month}{str(year)[-2:]} Curncy" 
           for code in currencies.values()
           for year in years
           for month in month_codes]

# Create a Bloomberg Historical Data Request
service = session.getService("//blp/refdata")
request = service.createRequest("HistoricalDataRequest")

# Add tickers to the request
for ticker in tickers:
    request.getElement("securities").appendValue(ticker)

# Add the fields we want
request.getElement("fields").appendValue("PX_LAST")
request.getElement("fields").appendValue("VOLUME")
request.getElement("fields").appendValue("OPEN_INT")

# Define the date range
request.set("startDate", "20000101")
request.set("endDate", dt.datetime.today().strftime('%Y%m%d')) # Set to today's date, can also set a specific end Date request.set("endDate", "20251404")
request.set("periodicitySelection", "DAILY")

# Send the request
session.sendRequest(request)

CorrelationId(flags=[valtype: 3, classid: 0, sz: 0], rawvalue.intValue=3)

###  Processing and Storing the Data

Once the data is received, we’ll organise it into three separate pandas DataFrames, One for **price**, One for **volume** and One for **open interest**

> **Note:** Depending on how many tickers and how long your date range is, the data request might take a few seconds to a couple of minutes to complete. Be patient — Bloomberg is pulling a lot of information behind the scenes!

In [8]:
# Prepare storage containers
price_data, volume_data, open_interest_data = {}, {}, {}

# Process incoming data
while True:
    ev = session.nextEvent()
    for msg in ev:
        if msg.hasElement("securityData"):
            sec_data = msg.getElement("securityData")
            ticker = sec_data.getElementAsString("security")
            field_data = sec_data.getElement("fieldData")
            for i in range(field_data.numValues()):
                date = field_data.getValue(i).getElementAsDatetime("date")
                px_last = field_data.getValue(i).getElementAsFloat("PX_LAST") if field_data.getValue(i).hasElement("PX_LAST") else None
                volume = field_data.getValue(i).getElementAsFloat("VOLUME") if field_data.getValue(i).hasElement("VOLUME") else None
                open_int = field_data.getValue(i).getElementAsFloat("OPEN_INT") if field_data.getValue(i).hasElement("OPEN_INT") else None
                
                # Store data
                price_data.setdefault(date, {})[ticker] = px_last
                volume_data.setdefault(date, {})[ticker] = volume
                open_interest_data.setdefault(date, {})[ticker] = open_int
    if ev.eventType() == blpapi.Event.RESPONSE:
        break

# Stop the session
session.stop()

True

### What's Happening Here?

Let’s break down what this block of code is doing step by step

- We first create **three empty dictionaries** to store the data: one each for `price_data`, `volume_data`, and `open_interest_data`.
- The loop `while True:` waits for Bloomberg to send back data and processes it **as it arrives**, event by event.
- Each event (`ev`) may contain multiple messages (`msg`), each of which holds data for one ticker.
- We extract:
  - The **ticker name** (e.g., `ECH10 Curncy`)
  - The **date** of each observation
  - The value for each field we requested: **PX_LAST**, **VOLUME**, **OPEN_INT**
- We use `.setdefault()` to group all tickers under each date, organizing it in a way that we can easily convert to a **DataFrame** later.
- Once Bloomberg finishes sending data (`Event.RESPONSE`), the loop breaks.
- Finally, we close the connection with `session.stop()`.

This gives us clean, structured dictionaries with **daily values for each contract**, ready to be transformed into pandas DataFrames.

## Step 5: Convert to DataFrames and Save to CSV

Now that we’ve collected all the data into Python dictionaries, we can convert each one into a **pandas DataFrame**. This gives us a clean tabular format that’s easy to analyse and work with.

We then save each DataFrame as a **CSV file** so the data can be reused or shared without needing to connect to Bloomberg again.

> This is especially useful if you're working on your analysis over multiple sessions — you don’t need to redownload everything every time.

In [18]:
# Convert dictionaries to pandas DataFrames
df_price = pd.DataFrame.from_dict(price_data, orient='index').sort_index()
df_volume = pd.DataFrame.from_dict(volume_data, orient='index').sort_index()
df_open_interest = pd.DataFrame.from_dict(open_interest_data, orient='index').sort_index()

# (Optional) Rename index if needed
df_price.index.name = 'Date'
df_volume.index.name = 'Date'
df_open_interest.index.name = 'Date'

Lets now look at the dataframe

In [19]:
df_price.head()

,BPH00 Curncy,BPM00 Curncy,BPU00 Curncy,BPZ00 Curncy,ECH00 Curncy,ECM00 Curncy,ECU00 Curncy,ECZ00 Curncy,JYH00 Curncy,JYM00 Curncy,...,SEH25 Curncy,NVM25 Curncy,NOM25 Curncy,SEM25 Curncy,NVU25 Curncy,NOU25 Curncy,SEU25 Curncy,NVZ25 Curncy,NOZ25 Curncy,SEZ25 Curncy
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,163.94,163.88,163.80,163.66,1.0329,1.0397,1.0458,1.0519,99.64,101.23,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-04,163.72,163.66,163.58,163.44,1.0352,1.0422,1.0487,1.0552,97.91,99.49,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,164.20,164.14,164.06,163.92,1.0369,1.0439,1.0504,1.0569,96.83,98.41,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-06,164.76,164.70,164.62,164.48,1.0349,1.0419,1.0484,1.0549,95.78,97.35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-07,163.86,163.80,163.72,163.58,1.0332,1.0403,1.0468,1.0533,95.96,97.53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


If you want them to be saved on csv files so that you can work on your personal computer you do the following

In [20]:
# Save each DataFrame to a CSV file
today = dt.datetime.today().strftime('%Y%m%d')
data_path = "../data/fx/futures"
df_price.to_csv(f'g10_futures_prices_wn_{today}.csv')
df_volume.to_csv(f'g10_futures_volume_wn_{today}.csv')
df_open_interest.to_csv(f'g10_futures_open_interest_wn_{today}.csv')

print("Data extraction complete and saved to CSV files.")

Data extraction complete and saved to CSV files.


## Final Notes: Bloomberg Limits, Support & Licensing

### Data Download Limits

Bloomberg applies download restrictions to academic licenses, and these limits can vary by institution.

- **Limits are enforced per machine** — if a computer reaches its data cap, your request may return no results.
- Try a **different Bloomberg Terminal** if you're having issues (e.g. another machine in the Business School).
- Limits typically reset every 24 hours.

> **Want to know the exact limit?**  
> Ask Bloomberg directly via the Terminal by pressing `Help` twice or typing `HELP <GO>`, then request a callback or start a chat.
> (The last time I called them they weren't able to tell me the exact limit so I am not entirely sure what the limit is but there is a limit.)

---

### Where to Get Help

You have several great support options if you get stuck:

#### Bloomberg Terminal Support
- Press `Help` twice or type `HELP <GO>` for live chat
- You can **request a callback**, and they’ll call you within minutes
- Available **24/7**, even for students

#### Imperial Library Support
- The **library staff are trained to help** with Bloomberg issues — don’t hesitate to ask!
- Find support and guides here:  
  🔗 [Bloomberg at Imperial College](https://www.imperial.ac.uk/admin-services/library/subject-support/business/bloomberg/)

---

### Data Usage & Licensing

Please note:
> **Bloomberg Terminal licenses include a data usage addendum.**  
> Data must be used **on the terminal where it was exported** and **cannot be copied to or redistributed through third-party tools** (like Google Sheets or cloud apps) on your personal device.

This is part of Bloomberg’s licensing terms and helps protect their commercial data services.

---

This wraps up your introduction to **programmatic Bloomberg data access** using Python! You're now ready to start exploring futures, equities, macroeconomic indicators and more.

## Bloomberg Ticker Conventions and Codes

When working with Bloomberg data, tickers follow specific naming conventions depending on the asset class. Below are examples and explanations for each category you'll commonly encounter:

---

### Currency Futures (Clearport Codes)

Currency futures are identified by **Clearport codes** and follow the format
[Currency Code][Month Code][Year Code] Curncy

- Example: `BPH25 Curncy` = British Pound March 2025 Futures Contract  

You can find the full list of Clearport codes here:  
🔗 [CME FX Futures Product Guide](https://www.cmegroup.com/markets/fx/fx-product-guide.html#futures)


---



### Spot Currency Pairs (Forex)

Spot FX tickers use the format
[BaseCurrency][QuoteCurrency] Curncy

- Example: `GBPUSD Curncy`, `EURCHF Curncy`, `USDCHF Curncy`  

---

### 10-Year Bond Futures

Bond futures follow a similar format, using **two-letter codes** and monthly expiries

[Bond Code] [Month Code][Year Code] Comdty

| Country     | Futures Type                  | Code | Example Ticker       |
|-------------|-------------------------------|------|-----------------------|
| UK          | Gilt Futures (ICE)            | G    | `G H20 Comdty`        |
| US          | 10Y Treasury Futures (CME)    | TY   | `TYH20 Comdty`        |
| Germany     | Bund Futures (Eurex)          | A    | `A H20 Comdty`        |
| Canada      | Govt Bond Futures (MX)        | CN   | `CNH20 Comdty`        |
| Japan       | JGB Futures (JPX)             | JB   | `JBH20 Comdty`        |
| Japan (ICE) | Alt JGB Futures (ICE)         | N    | `N H20 Comdty`        |


---

### Equity Index Futures

Index futures use the following format

[Index Code][Month Code][Year Code] Index

| Index            | Code | Example Ticker     |
|------------------|------|--------------------|
| EURO STOXX 50    | VG   | `VGU24 Index`      |
| DOW JONES        | DM   | `DMU24 Index`      |
| S&P 500          | ES   | `ESU24 Index`      |
| NASDAQ           | NQ   | `NQU24 Index`      |
| FTSE 100         | Z    | `Z U24 Index`      |
| DAX              | GX   | `GXU24 Index`      |
| CAC              | CF   | `CFU24 Index`      |
| IBEX 35          | IB   | `IBU24 Index`      |
| ASX              | XP   | `XPU24 Index`      |
| NIKKEI           | NK   | `NKU24 Index`      |
| TOPIX            | TP   | `TPU24 Index`      |
| KOSPI            | KM   | `KMU24 Index`      |
| HANG SENG        | HI   | `HIU24 Index`      |
| S&P TSX 60       | PT   | `PTU24 Index`      |


---


### Major Stock Indices (Spot Levels)

For daily index levels (not futures), use the format:

[Index Symbol] Index

| Index Name     | Bloomberg Code |
|----------------|----------------|
| S&P 500        | SPX Index      |
| Dow Jones      | INDU Index     |
| NASDAQ         | CCMP Index     |
| EURO STOXX 50  | SX5E Index     |
| FTSE 100       | UKX Index      |
| DAX            | DAX Index      |
| NIKKEI 225     | NKY Index      |
| HANG SENG      | HSI Index      |
| CAC 40         | CAC Index      |
| IBEX 35        | IBEX Index     |
| KOSPI          | KOSPI Index    |
| TOPIX          | TPX Index      |
| S&P/TSX 60     | SPTSX Index    |


---

### Month Codes for Futures

All futures contracts follow a standard expiration code for each month:

| Month     | Code |
|-----------|------|
| March     | H    |
| June      | M    |
| September | U    |
| December  | Z    |

---

This structure helps you build tickers programmatically — just match the right code, month, and year, and add the correct suffix (`Curncy`, `Comdty`, or `Index`).